# Transformer 번역 실습 (Attention Is All You Need)

작업 방식은 두 단계입니다.

- **로컬 (`MODE = "local"`)**: 짧은 문장 몇 개(`TINY_TEXT`)로 shape, 학습, 추론이 돌아가는지만 확인합니다. 같은 문장으로 학습해서 그대로 번역해내면(overfit test) 코드가 맞는 겁니다.
- **Colab (`MODE = "colab"`)**: 같은 노트북을 올려서 Multi30k(EN→DE)로 본 학습, BLEU 평가를 합니다.

`MODE`만 바꾸면 되고, 모델 코드(Part B)는 두 환경에서 동일합니다.


In [21]:
import math
import random
import re
from collections import Counter

import torch
import torch.nn as nn

PAD, BOS, EOS, UNK = "<pad>", "<bos>", "<eos>", "<unk>"
SPECIALS = [PAD, BOS, EOS, UNK]

MODE = "local"   # "local": 동작 확인 / "colab": Multi30k 본 학습
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 시작점으로 잡은 값들이므로 자유롭게 바꿔가며 실험하세요.
CONFIG = {
    "local": dict(N=2, d_model=64, h=4, d_ff=128, dropout=0.0,
                  epochs=200, batch_size=8, warmup=50, max_vocab=None, min_freq=1, max_len=30),
    "colab": dict(N=3, d_model=256, h=8, d_ff=512, dropout=0.1,
                  epochs=15, batch_size=128, warmup=800, max_vocab=8000, min_freq=2, max_len=30),
}[MODE]
print(MODE, DEVICE)

local cuda


## Part A. 데이터: 문장 쌍 -> 토큰 -> 어휘 -> 텐서

In [22]:
def tokenize(text):
    """소문자로 바꾸고 단어/구두점 단위로 분리."""
    return re.findall(r"\w+|[^\w\s]", text.lower())


def to_token_pairs(text_pairs, max_len=30):
    """[(영어 문장, 독일어 문장), ...] -> [(src_tokens, tgt_tokens), ...]. 너무 긴 문장은 제외."""
    pairs = []
    for en, de in text_pairs:
        s, t = tokenize(en), tokenize(de)
        if 0 < len(s) <= max_len and 0 < len(t) <= max_len:
            pairs.append((s, t))
    return pairs

In [23]:
# 로컬 동작 확인용 초소형 데이터 (EN -> DE)
TINY_TEXT = [
    ("a man is running .", "ein mann läuft ."),
    ("a dog is sleeping .", "ein hund schläft ."),
    ("two children are playing .", "zwei kinder spielen ."),
    ("a woman is reading a book .", "eine frau liest ein buch ."),
    ("the cat sits on the table .", "die katze sitzt auf dem tisch ."),
    ("a boy is eating an apple .", "ein junge isst einen apfel ."),
    ("a girl is singing .", "ein mädchen singt ."),
    ("the man reads a newspaper .", "der mann liest eine zeitung ."),
]


def load_multi30k(max_len=30):
    """Colab 전용. 먼저 `!pip install datasets` 필요. (train, val, test) 토큰 쌍 반환."""
    from datasets import load_dataset

    ds = load_dataset("bentrevett/multi30k")
    val_key = "validation" if "validation" in ds else "val"

    def convert(split):
        return to_token_pairs([(r["en"], r["de"]) for r in ds[split]], max_len)

    return convert("train"), convert(val_key), convert("test")


def get_data():
    if MODE == "local":
        pairs = to_token_pairs(TINY_TEXT, CONFIG["max_len"])
        return pairs, pairs, pairs   # 같은 데이터로 학습/평가 -> 외웠는지(overfit) 확인
    return load_multi30k(CONFIG["max_len"])

In [24]:
def build_vocab(pairs, max_size=None, min_freq=1):
    """src/tgt 어휘를 따로 구축. SPECIALS가 맨 앞 (PAD=0), 나머지는 빈도순.
    min_freq 미만은 제외, max_size가 있으면 상위 max_size개만. 반환: (src_stoi, tgt_stoi)"""
    def make(token_lists):
        counter = Counter(w for toks in token_lists for w in toks)
        items = sorted(counter.items(), key=lambda kv: (-kv[1], kv[0]))
        words = [w for w, c in items if c >= min_freq]
        if max_size is not None:
            words = words[:max_size]
        return {w: i for i, w in enumerate(SPECIALS + words)}

    return make([s for s, _ in pairs]), make([t for _, t in pairs])


def encode(tokens, stoi, add_bos_eos=False):
    """토큰 -> id. 모르는 토큰은 UNK. add_bos_eos면 앞뒤에 BOS/EOS."""
    ids = [stoi.get(t, stoi[UNK]) for t in tokens]
    if add_bos_eos:
        ids = [stoi[BOS]] + ids + [stoi[EOS]]
    return ids


def collate(batch_pairs, src_stoi, tgt_stoi, device="cpu"):
    """문장 쌍 리스트 -> 패딩된 (src, tgt) LongTensor. src는 BOS/EOS 없음, tgt는 있음."""
    src_ids = [encode(s, src_stoi) for s, _ in batch_pairs]
    tgt_ids = [encode(t, tgt_stoi, add_bos_eos=True) for _, t in batch_pairs]

    def pad_to_tensor(seqs, pad_id):
        max_len = max(len(s) for s in seqs)
        return torch.tensor([s + [pad_id] * (max_len - len(s)) for s in seqs], device=device)

    return pad_to_tensor(src_ids, src_stoi[PAD]), pad_to_tensor(tgt_ids, tgt_stoi[PAD])


def ids_to_tokens(ids, stoi):
    """id 리스트 -> 토큰 리스트. EOS에서 멈추고 BOS/PAD는 제외."""
    itos = {i: w for w, i in stoi.items()}
    out = []
    for i in ids:
        w = itos[int(i)]
        if w == EOS:
            break
        if w not in (BOS, PAD):
            out.append(w)
    return out

### 데이터 확인

In [25]:
train_pairs, val_pairs, test_pairs = get_data()
src_stoi, tgt_stoi = build_vocab(train_pairs, CONFIG["max_vocab"], CONFIG["min_freq"])
src, tgt = collate(train_pairs[:3], src_stoi, tgt_stoi)
print(len(train_pairs), len(val_pairs), len(test_pairs))
print('vocab:', len(src_stoi), len(tgt_stoi))
print(train_pairs[0])
print(src)
print(tgt)

8 8 8
vocab: 31 31
(['a', 'man', 'is', 'running', '.'], ['ein', 'mann', 'läuft', '.'])
tensor([[ 5,  8,  6, 24,  4],
        [ 5, 16,  6, 27,  4],
        [29, 15, 11, 21,  4]])
tensor([[ 1,  5,  8, 22,  4,  2],
        [ 1,  5, 17, 24,  4,  2],
        [ 1, 30, 21, 27,  4,  2]])


## Part B. 모델 (직접 구현)

In [26]:
def make_pad_mask(seq, pad_id=0):
    """(B, L) -> (B, 1, L) bool. 진짜 토큰 True."""
    pad = (seq != pad_id)
    pad = pad.unsqueeze(-2)
    return pad

In [27]:
def subsequent_mask(size):
    """(1, size, size) bool. 미래 위치는 False (하삼각 True)."""
    sub_mask = torch.ones(size, size)
    sub_mask = torch.tril(sub_mask)
    sub_mask = sub_mask.bool().unsqueeze(0)
    return sub_mask

In [28]:
def make_tgt_mask(tgt, pad_id=0):
    """pad 마스크 & subsequent 마스크 -> (B, L, L)."""
    pad_mask = make_pad_mask(tgt)
    sub_mask = subsequent_mask(tgt.shape[-1])
    return pad_mask & sub_mask

### 마스크 확인

In [29]:
tgt_in = torch.tensor([[1, 5, 8, 0], [1, 6, 0, 0]])   # 0이 pad
print(make_pad_mask(tgt_in))        # (2, 1, 4)
print(subsequent_mask(4)[0].int())        # 하삼각
print(make_tgt_mask(tgt_in).shape)        # (2, 4, 4)
print(make_tgt_mask(tgt_in))     # pad 열이 0인 하삼각

tensor([[[ True,  True,  True, False]],

        [[ True,  True, False, False]]])
tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 1]], dtype=torch.int32)
torch.Size([2, 4, 4])
tensor([[[ True, False, False, False],
         [ True,  True, False, False],
         [ True,  True,  True, False],
         [ True,  True,  True, False]],

        [[ True, False, False, False],
         [ True,  True, False, False],
         [ True,  True, False, False],
         [ True,  True, False, False]]])


> **Colab(GPU) 주의**: `subsequent_mask`는 CPU에서 만들어지므로, `make_tgt_mask`에서 pad 마스크(GPU)와 `&` 할 때 device가 달라 에러가 납니다. `subsequent_mask(...)`를 `tgt.device`로 옮겨서(`.to(tgt.device)`) 결합하세요.

In [30]:
def attention(q, k, v, mask=None, dropout=None):
    """softmax(QK^T / sqrt(d_k)) V.  (out, attn_weights) 반환."""
    d_k = q.size(-1)
    E = q @ k.transpose(-2,-1) / math.sqrt(d_k)
    if mask is not None: E = E.masked_fill(~mask, -1e9)
    A = E.softmax(-1)
    if dropout is not None: A = dropout(A)
    return A @ v, A

### self-attention vs cross-attention 마스킹 확인 (임의의 값)

In [31]:
d = 4
src = torch.tensor([[1, 2, 3, 0, 0],      # 문장 A: 실제 토큰 3개 + pad 2개
                     [1, 2, 3, 4, 5]])    # 문장 B: pad 없음
tgt = torch.tensor([[1, 2, 3,0,0,0],            # 디코더 쪽 길이는 src와 다르게 3
                     [1, 2, 3,4,0,0]])

src_mask = make_pad_mask(src)             # (B, 1, L_src)
src_embed = torch.randn(2, src.shape[1], d)   # 인코더 입력 자리에 넣을 임의의 값
tgt_embed = torch.randn(2, tgt.shape[1], d)   # 디코더 입력 자리에 넣을 임의의 값
tgt_mask = make_tgt_mask(tgt)

print('src_mask:', src_mask.shape)
print(src_mask)


# self-attention: q=k=v=src_embed, L_q=L_k=5 (길이 같아도 pad 있으면 마스킹 필요)
_, A_self_d = attention(tgt_embed, tgt_embed, tgt_embed, tgt_mask)
print()
print('[self-attention] 문장 A weight (마지막 2열=pad, 0이어야 정상):')
print(A_self_d)

src_mask: torch.Size([2, 1, 5])
tensor([[[ True,  True,  True, False, False]],

        [[ True,  True,  True,  True,  True]]])

[self-attention] 문장 A weight (마지막 2열=pad, 0이어야 정상):
tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1666, 0.8334, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1475, 0.3728, 0.4798, 0.0000, 0.0000, 0.0000],
         [0.2095, 0.4043, 0.3862, 0.0000, 0.0000, 0.0000],
         [0.2511, 0.2711, 0.4778, 0.0000, 0.0000, 0.0000],
         [0.9182, 0.0486, 0.0332, 0.0000, 0.0000, 0.0000]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2135, 0.7865, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0010, 0.0049, 0.9941, 0.0000, 0.0000, 0.0000],
         [0.0076, 0.0745, 0.3477, 0.5701, 0.0000, 0.0000],
         [0.1085, 0.0549, 0.7619, 0.0748, 0.0000, 0.0000],
         [0.4326, 0.3625, 0.1663, 0.0387, 0.0000, 0.0000]]])


In [32]:
class MultiHeadAttention(nn.Module):
    def __init__(self, h, d_model, p=0.1):
        super().__init__()
        self.d_k = d_model // h
        self.h = h
        self.linear_q = nn.Linear(d_model, d_model)
        self.linear_k = nn.Linear(d_model, d_model)
        self.linear_v = nn.Linear(d_model, d_model)
        self.linear_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(p)

    def forward(self, q, k, v, mask=None):
        if mask is not None: mask = mask.unsqueeze(1)
        B, L_q, _ = q.shape
        q = self.linear_q(q)
        q = q.reshape(B, L_q, self.h, self.d_k).transpose(1,2)
        
        _, L_K, _ = k.shape
        k = self.linear_k(k)
        k = k.reshape(B, L_K, self.h, self.d_k).transpose(1,2)
        v = self.linear_v(v)
        v = v.reshape(B, L_K, self.h, self.d_k).transpose(1,2)
        
        out, _ = attention(q,k,v,mask,self.dropout)
        out = out.transpose(1,2).reshape(B, L_q, self.h * self.d_k)
        
        return self.linear_o(out)

In [33]:
class PositionwiseFFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model,d_ff)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff,d_model)

    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)

        return x

In [34]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=100):
        super().__init__()
        self.d_model = d_model
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("PE_mat", torch.zeros(max_len, d_model))
        for pos in range(max_len):
            for i in range(self.d_model//2):
                self.PE_mat[pos, 2*i] = math.sin(pos/10000**(2*i/self.d_model))
                self.PE_mat[pos, 2*i+1] = math.cos(pos/10000**(2*i/self.d_model))

    def forward(self, x):
        L = x.shape[1]
        PE_mat_cut = self.PE_mat[:L, :]
        return self.dropout(PE_mat_cut).unsqueeze(0)


In [35]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, h, d_ff, dropout):
        super().__init__()
        self.MHSA = MultiHeadAttention(h, d_model, dropout)
        self.ffn = PositionwiseFFN(d_model, d_ff, dropout)
        self.layernorm1 = nn.LayerNorm(d_model)
        self.layernorm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        x = self.layernorm1(x + self.dropout(self.MHSA(x,x,x,src_mask)))
        x = self.layernorm2(x + self.dropout(self.ffn(x)))
        return x

In [36]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, h, d_ff, dropout):
        super().__init__()
        self.MMHSA = MultiHeadAttention(h, d_model, dropout)
        self.dropout = nn.Dropout(dropout)
        self.layernorm1 = nn.LayerNorm(d_model)
        self.MHCA = MultiHeadAttention(h, d_model, dropout)
        self.layernorm2 = nn.LayerNorm(d_model)
        self.layernorm3 = nn.LayerNorm(d_model)
        self.ffn = PositionwiseFFN(d_model, d_ff, dropout)

    def forward(self, x, memory, src_mask, tgt_mask):
        x = self.layernorm1(x + self.dropout(self.MMHSA(x,x,x,tgt_mask)))
        x = self.layernorm2(x + self.dropout(self.MHCA(x,memory,memory,src_mask)))
        x = self.layernorm3(x + self.dropout(self.ffn(x)))
        return x

In [37]:
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, N=6, d_model=512, h=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.src_embed = nn.Embedding(src_vocab, d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab, d_model)
        self.pos_encode = PositionalEncoding(d_model)
        self.EncoderBlock = nn.ModuleList([EncoderLayer(d_model, h, d_ff, dropout) for _ in range(N)])
        self.DecoderBlock = nn.ModuleList([DecoderLayer(d_model, h, d_ff, dropout) for _ in range(N)])
        self.last_linear = nn.Linear(d_model, tgt_vocab)
        self.d_model = d_model

    def encode(self, src, src_mask):
        memory = self.src_embed(src) * math.sqrt(self.d_model)
        memory = memory + self.pos_encode(memory)
        for layer in self.EncoderBlock: memory = layer(memory, src_mask)
        return memory

    def decode(self, memory, src_mask, tgt, tgt_mask):
        x = self.tgt_embed(tgt) * math.sqrt(self.d_model)
        x = x + self.pos_encode(x)
        for layer in self.DecoderBlock: x = layer(x, memory, src_mask, tgt_mask)
        return x

    def forward(self, src, tgt_in):
        """src, tgt_in (BOS로 시작, EOS 제외) -> logits (B, L, tgt_vocab)."""
        src_mask = make_pad_mask(src)
        tgt_mask = make_tgt_mask(tgt_in)
        memory = self.encode(src, src_mask)
        out = self.decode(memory, src_mask, tgt_in, tgt_mask)
        return self.last_linear(out)

### Transformer shape 확인

In [38]:
model = Transformer(src_vocab=len(src_stoi), tgt_vocab=len(tgt_stoi))
logits = model(src, tgt[:, :-1])
print(logits.shape)   # (3, tgt_len-1, tgt_vocab)

torch.Size([2, 5, 31])


## Part C. 학습

In [39]:
def train(model, train_pairs, val_pairs, src_stoi, tgt_stoi, epochs, batch_size, warmup):
    """teacher forcing 학습 루프.
    - 매 epoch마다 train_pairs를 random.shuffle -> batch_size씩 잘라 collate(..., device=DEVICE)
    - 디코더 입력 tgt[:, :-1], 정답 tgt[:, 1:]
    - loss: CrossEntropyLoss(ignore_index=PAD id), logits.reshape(-1, V) / y.reshape(-1)
    - 옵티마이저: Adam(betas=(0.9, 0.98), eps=1e-9) + 논문 5.3절 warmup 스케줄(LambdaLR)
    - model.train()/model.eval() 구분, val loss 계산은 no_grad로
    - 에폭마다 train loss(와 val loss) 출력
    """
    pad_id = tgt_stoi[PAD]
    loss_fn = nn.CrossEntropyLoss(ignore_index=pad_id)
    optimizer = torch.optim.Adam(model.parameters(), betas=(0.9, 0.98), eps=1e-9)

    def lr_lambda(step):
        step = max(step, 1)   # step=0일 때 0**-0.5 (0으로 나누기) 방지
        return model.d_model ** -0.5 * min(step ** -0.5, step * warmup ** -1.5)

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    for epoch in range(epochs):
        # ---- 1) 학습 ----
        model.train()
        random.shuffle(train_pairs)
        train_loss_sum, train_n_tokens = 0.0, 0

        for i in range(0, len(train_pairs), batch_size):
            batch = train_pairs[i:i + batch_size]
            src, tgt = collate(batch, src_stoi, tgt_stoi, device=DEVICE)
            tgt_in, y = tgt[:, :-1], tgt[:, 1:]

            logits = model(src, tgt_in)
            V = logits.shape[-1]
            loss = loss_fn(logits.reshape(-1, V), y.reshape(-1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

            n_tokens = (y != pad_id).sum().item()
            train_loss_sum += loss.item() * n_tokens
            train_n_tokens += n_tokens

        train_loss = train_loss_sum / train_n_tokens

        # ---- 2) 검증 ----
        model.eval()
        val_loss_sum, val_n_tokens = 0.0, 0
        with torch.no_grad():
            for i in range(0, len(val_pairs), batch_size):
                batch = val_pairs[i:i + batch_size]
                src, tgt = collate(batch, src_stoi, tgt_stoi, device=DEVICE)
                tgt_in, y = tgt[:, :-1], tgt[:, 1:]

                logits = model(src, tgt_in)
                V = logits.shape[-1]
                loss = loss_fn(logits.reshape(-1, V), y.reshape(-1))

                n_tokens = (y != pad_id).sum().item()
                val_loss_sum += loss.item() * n_tokens
                val_n_tokens += n_tokens

        val_loss = val_loss_sum / val_n_tokens

        print(f"epoch {epoch+1}/{epochs}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

### 로컬 과적합 테스트 (MODE="local")

loss가 0에 가깝게 내려가야 합니다. 안 내려가면 모델/마스크에 버그가 있는 겁니다.

In [41]:
torch.manual_seed(0)
model = Transformer(len(src_stoi), len(tgt_stoi), N=CONFIG["N"], d_model=CONFIG["d_model"],
                    h=CONFIG["h"], d_ff=CONFIG["d_ff"], dropout=CONFIG["dropout"]).to(DEVICE)
train(model, train_pairs, val_pairs, src_stoi, tgt_stoi, CONFIG["epochs"], CONFIG["batch_size"], CONFIG["warmup"])

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

## Part D. 추론 & 평가

In [ ]:
@torch.no_grad()
def greedy_decode(model, src, src_stoi, tgt_stoi, max_len=30):
    """src: (1, L) 텐서(DEVICE 위). encode는 한 번만, decode는 BOS부터 한 토큰씩 반복.
    가장 확률 높은 토큰을 이어붙이다 EOS가 나오거나 max_len이 되면 중단. tgt id 리스트 반환."""
    raise NotImplementedError

In [ ]:
def evaluate(model, pairs, src_stoi, tgt_stoi, n_show=5):
    """model.eval() 후 각 문장을 greedy_decode로 번역해 정답과 비교.
    - local: 문장 단위 exact match 비율 (과적합 sanity check, 1.0이 나와야 함)
    - colab: sacrebleu.corpus_bleu(가설 문장들, [정답 문장들]).score  (`!pip install sacrebleu`)
    - 예시 n_show개를 src / 정답 / 모델 출력 순으로 출력 (ids_to_tokens 사용)
    """
    raise NotImplementedError

### 실행

In [ ]:
def main():
    torch.manual_seed(0)
    train_pairs, val_pairs, test_pairs = get_data()
    src_stoi, tgt_stoi = build_vocab(train_pairs, CONFIG["max_vocab"], CONFIG["min_freq"])
    model = Transformer(len(src_stoi), len(tgt_stoi), N=CONFIG["N"], d_model=CONFIG["d_model"],
                        h=CONFIG["h"], d_ff=CONFIG["d_ff"], dropout=CONFIG["dropout"]).to(DEVICE)
    train(model, train_pairs, val_pairs, src_stoi, tgt_stoi,
          CONFIG["epochs"], CONFIG["batch_size"], CONFIG["warmup"])
    evaluate(model, test_pairs, src_stoi, tgt_stoi)

In [ ]:
# main()